# H3 compact

Step-by-step animation of [`h3compact`](https://github.com/opengeoshub/vgrid/blob/main/vgrid/conversion/dggscompact/h3compact.py) / `h3.compact_cells`.

Input: [`h3_10.geojson`](h3_10.geojson) in this folder.

Mirrors upstream **`compactCells`** in [h3lib `h3Index.c`](https://github.com/uber/h3/blob/master/src/h3lib/lib/h3Index.c) (via `h3-py` → `h3lib.compactCells`):

1. Input cells must share the **same resolution** (see `h3.compact_cells` docs).
2. Each **round** takes the current working set at resolution `r`, groups by **`cell_to_parent(cell, r - 1)`**, and finds parents whose children equal **`cell_to_children(parent)`** (7 hex children, **6 for pentagon** parents — H3 counts the deleted pentagon child implicitly).
3. **Complete sibling sets** merge into their parent; **incomplete** cells are finalized (they appear in the compact output as-is).
4. The working set becomes the new parents; repeat until resolution 0 or no merges.

Frame **3a** highlights cyan children and orange **parent** (dashed). The final frame matches `h3.compact_cells(input)`.

## Install necessary packages

In [ ]:
%pip install vgrid geopandas matplotlib imageio pillow h3
# optional for MP4:
%pip install imageio-ffmpeg

In [ ]:
"""Step-by-step h3 compact animation."""
from collections import defaultdict
from pathlib import Path

import geopandas as gpd
import h3
import imageio.v2 as imageio
import matplotlib.pyplot as plt
from matplotlib.collections import PatchCollection
from matplotlib.patches import Polygon as MplPolygon

from vgrid.conversion.dggs2geo.h32geo import h32geo

INPUT_GEOJSON = Path("h3_10.geojson")
H3_ID_FIELD = "h3"
OUT_GIF = "h3_compact.gif"
OUT_MP4 = "h3_compact.mp4"
FRAME_EVERY_MERGE = 1
CHILDREN_LABEL = "children (merge group)"  # 7 hex or 6 pentagon
DPI = 120
FIX_ANTIMERIDIAN = None


def child_count_label(parent_id):
    n = len(h3.cell_to_children(parent_id))
    kind = "pentagon" if h3.is_pentagon(parent_id) else "hex"
    return f"{n} {kind} children"


def cell_patches(cell_polys, facecolor, edgecolor, alpha=0.85, lw=1.0):
    patches = []
    for poly in cell_polys:
        if poly is None or poly.is_empty:
            continue
        patches.append(MplPolygon(list(poly.exterior.coords), closed=True))
    return PatchCollection(
        patches,
        facecolor=facecolor,
        edgecolor=edgecolor,
        alpha=alpha,
        linewidths=lw,
        zorder=2,
    )


def polys_for_ids(cell_ids, fix_antimeridian=None):
    polys = []
    for cell_id in cell_ids:
        poly = h32geo(cell_id, fix_antimeridian=fix_antimeridian)
        if poly is not None and not poly.is_empty:
            polys.append(poly)
    return polys


def render_frame(
    bounds,
    title,
    path,
    background_polys=None,
    child_polys=None,
    parent_poly=None,
):
    fig, ax = plt.subplots(figsize=(8, 8))
    minx, miny, maxx, maxy = bounds
    pad = max(maxx - minx, maxy - miny) * 0.06 or 0.01
    ax.set_xlim(minx - pad, maxx + pad)
    ax.set_ylim(miny - pad, maxy + pad)

    if background_polys:
        ax.add_collection(
            cell_patches(background_polys, "#e8eaf6", "#5c6bc0", alpha=0.5, lw=0.8)
        )
    if child_polys:
        ax.add_collection(
            cell_patches(child_polys, "#00bcd4", "#006064", alpha=0.9, lw=1.4)
        )
    if parent_poly is not None:
        ax.add_collection(
            cell_patches([parent_poly], "#ff9800", "#e65100", alpha=0.55, lw=2.0)
        )
        gpd.GeoSeries([parent_poly.boundary]).plot(
            ax=ax, color="#e65100", lw=3.5, linestyle="--", zorder=4
        )

    ax.set_facecolor("#fafafa")
    ax.plot([], [], color="#00bcd4", lw=4, label=CHILDREN_LABEL)
    ax.plot([], [], color="#ff9800", lw=4, label="parent cell")
    ax.plot([], [], color="#5c6bc0", lw=4, label="other cells")
    ax.legend(loc="upper right", fontsize=8, framealpha=0.95)
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.grid(False)
    fig.subplots_adjust(left=0.08, right=0.92, top=0.92, bottom=0.08)
    fig.savefig(path, dpi=DPI, facecolor="white")
    plt.close(fig)


def compact_one_round(current_ids):
    """One outer-loop iteration of h3lib compactCells (h3Index.c).

    At resolution r, group by cell_to_parent(·, r-1). Parents with a full
    child set merge; other cells are finalized for output.
    """
    if not current_ids:
        return set(), set(), []

    res = h3.get_resolution(next(iter(current_ids)))
    if res == 0:
        return set(), set(current_ids), []

    parent_res = res - 1
    grouped = defaultdict(set)
    for cell_id in current_ids:
        grouped[h3.cell_to_parent(cell_id, parent_res)].add(cell_id)

    merges = []
    compactable = set()
    for parent, children in grouped.items():
        if children == set(h3.cell_to_children(parent)):
            compactable.add(parent)
            merges.append((parent, sorted(children)))

    merges.sort(key=lambda item: item[0])  # stable sweep order by parent H3 index

    remaining = set()
    finalized = set()
    for cell_id in current_ids:
        parent = h3.cell_to_parent(cell_id, parent_res)
        if parent in compactable:
            remaining.add(parent)
        else:
            finalized.add(cell_id)

    return remaining, finalized, merges


def validate_same_resolution(cell_ids):
    resolutions = {h3.get_resolution(c) for c in cell_ids}
    if len(resolutions) > 1:
        raise ValueError(
            f"Input cells must share the same resolution; got {sorted(resolutions)}"
        )


def h3_compact_with_frames(h3_ids, bounds, frame_dir, fix_antimeridian=None):
    frame_dir.mkdir(parents=True, exist_ok=True)
    frames = []
    idx = 0
    id_to_poly = {}

    def poly_for(cell_id):
        if cell_id not in id_to_poly:
            id_to_poly[cell_id] = h32geo(cell_id, fix_antimeridian=fix_antimeridian)
        return id_to_poly[cell_id]

    def snap_simple(polys, title):
        nonlocal idx
        p = frame_dir / f"frame_{idx:04d}.png"
        render_frame(bounds, title, p, background_polys=polys)
        frames.append(p)
        idx += 1

    def snap_merge(current_ids, parent_id, child_ids, title):
        nonlocal idx
        child_set = set(child_ids)
        background = [
            poly_for(cid)
            for cid in current_ids
            if cid not in child_set
            and poly_for(cid) is not None
            and not poly_for(cid).is_empty
        ]
        children = [
            poly_for(cid)
            for cid in child_ids
            if poly_for(cid) is not None and not poly_for(cid).is_empty
        ]
        parent_poly = poly_for(parent_id)
        p = frame_dir / f"frame_{idx:04d}.png"
        render_frame(
            bounds,
            title,
            p,
            background_polys=background,
            child_polys=children,
            parent_poly=parent_poly,
        )
        frames.append(p)
        idx += 1

    input_polys = [
        p
        for cid in h3_ids
        for p in [poly_for(cid)]
        if p is not None and not p.is_empty
    ]
    snap_simple(input_polys, f"1. Input grid ({len(h3_ids)} cells, res {h3.get_resolution(h3_ids[0])})")

    current = set(h3_ids)
    finalized = set()
    round_idx = 0
    while current:
        res = h3.get_resolution(next(iter(current)))
        if res == 0:
            finalized.update(current)
            snap_simple(
                polys_for_ids(sorted(finalized), fix_antimeridian),
                f"3c. Resolution-0 cells finalized ({len(current)} cells)",
            )
            break

        remaining, newly_finalized, merges = compact_one_round(current)
        if not merges:
            finalized.update(current)
            break

        round_idx += 1
        for merge_i, (parent_id, child_ids) in enumerate(merges, start=1):
            if merge_i % FRAME_EVERY_MERGE != 0 and merge_i != len(merges):
                continue
            snap_merge(
                current,
                parent_id,
                child_ids,
                f"3a. Round {round_idx} res {res}→{res - 1} merge {merge_i}/{len(merges)}: "
                f"{child_count_label(parent_id)} → {parent_id}",
            )

        finalized.update(newly_finalized)
        current = remaining
        snap_simple(
            polys_for_ids(sorted(finalized | current), fix_antimeridian),
            f"3b. After round {round_idx}: {len(finalized)} finalized, "
            f"{len(current)} still compacting",
        )

    animated_ids = sorted(finalized | current)
    ref_ids = sorted(h3.compact_cells(h3_ids))
    snap_simple(
        polys_for_ids(ref_ids, fix_antimeridian),
        f"4. Final compact set ({len(ref_ids)} cells) — h3.compact_cells",
    )
    if set(animated_ids) != set(ref_ids):
        print(
            "Warning: step-through differs from h3.compact_cells:",
            len(animated_ids),
            "vs",
            len(ref_ids),
        )
    else:
        print(f"Verified: animation matches h3.compact_cells ({len(ref_ids)} cells)")
    return frames, ref_ids


def main():
    print(f"Using {INPUT_GEOJSON}")
    gdf = gpd.read_file(INPUT_GEOJSON)
    h3_ids = gdf[H3_ID_FIELD].drop_duplicates().tolist()
    if not h3_ids:
        raise ValueError(f"No '{H3_ID_FIELD}' values in {INPUT_GEOJSON}")
    validate_same_resolution(h3_ids)

    bounds = gdf.total_bounds
    frame_dir = Path("_h3_compact_frames")
    frames, final_ids = h3_compact_with_frames(
        h3_ids, bounds, frame_dir, fix_antimeridian=FIX_ANTIMERIDIAN
    )
    GIF_FRAME_DURATION = 1.8
    imageio.mimsave(
        OUT_GIF, [imageio.imread(f) for f in frames], duration=GIF_FRAME_DURATION
    )
    print(
        f"Wrote {OUT_GIF} ({len(frames)} frames, "
        f"{len(h3_ids)} → {len(final_ids)} cells)"
    )
    try:
        MP4_FPS = 2.4
        writer = imageio.get_writer(OUT_MP4, fps=MP4_FPS)
        for f in frames:
            writer.append_data(imageio.imread(f))
        writer.close()
        print(f"Wrote {OUT_MP4}")
    except Exception as e:
        print(f"MP4 skipped ({e}). GIF is enough.")


if __name__ == "__main__":
    main()

Using h3_11.geojson
Verified: animation matches h3.compact_cells (367 cells)
Wrote h3_compact.gif (301 frames, 2143 → 367 cells)
Wrote h3_compact.mp4
